In [ ]:
  # -*- coding: utf-8 -*-
# ==============================================================================
#  Comprehensive Indic Script Transliteration Tool - VERSION 2.0 (MERGED & COMPLETE)
#
#  Method: Hub & Spoke with Contextual Post-Processing
#  - Devanagari as the central hub.
#  - NEW: Post-processing rules for orthographic accuracy (Schwa Deletion).
#  - NEW: Exception dictionary for handling loanwords and special cases.
# ==============================================================================

import re

# ---------------------------
# Unicode Ranges for Scripts
# ---------------------------
SCRIPT_RANGES = {
    "Telugu": (0x0C00, 0x0C7F),
    "Devanagari": (0x0900, 0x097F),
    "Tamil": (0x0B80, 0x0BFF),
    "Malayalam": (0x0D00, 0x0D7F),
    "Gurmukhi": (0x0A00, 0x0A7F),
    "Kannada": (0x0C80, 0x0CFF),
    "Bengali": (0x0980, 0x09FF),  # Also covers Assamese
    "Gujarati": (0x0A80, 0x0AFF),
    "Odia": (0x0B00, 0x0B7F),
    # "English": (0x0000, 0x007F),  # Basic Latin
}

# -----------------------------------------------------
# NEW: EXCEPTION DICTIONARY for loanwords and special cases.
# The script will check this dictionary first. All keys should be lowercase.
# -----------------------------------------------------
EXCEPTION_DICTIONARY = {
    "lipisathi": {"Devanagari": "लिपिसाथी", "Telugu": "లిపిసాథి"},
    "google": {"Devanagari": "गूगल", "Telugu": "గూగుల్"},
    # Add more exceptions here as you find them
}


def detect_script(text):
    """Detects the script of the given text based on character counts."""
    char_counts = {script: 0 for script in SCRIPT_RANGES}
    for char in text:
        code = ord(char)
        for script, (start, end) in SCRIPT_RANGES.items():
            if start <= code <= end:
                char_counts[script] += 1
                break
    if any(char_counts.values()):
        return max(char_counts, key=char_counts.get)
    return "Unknown"

# -----------------------------------------------------
# MAPPINGS: FROM EVERY SCRIPT TO DEVANAGARI (THE HUB)
# -- This section is now complete with all your original mappings --
# -----------------------------------------------------
MAPPINGS_TO_DEVANAGARI = {}

MAPPINGS_TO_DEVANAGARI["Telugu"] = {
    "అ":"अ", "ఆ":"आ", "ఇ":"इ", "ఈ":"ई", "ఉ":"उ", "ఊ":"ऊ", "ఋ":"ऋ", "ౠ":"ॠ",
    "ఎ":"ए", "ఏ":"ए", "ఐ":"ऐ", "ఒ":"ओ", "ఓ":"ओ", "ఔ":"औ",
    "క":"क", "ఖ":"ख", "గ":"ग", "ఘ":"घ", "ఙ":"ङ", "చ":"च", "ఛ":"छ", "జ":"ज", "ఝ":"झ", "ఞ":"ञ",
    "ట":"ट", "ఠ":"ठ", "డ":"ड", "ఢ":"ढ", "ణ":"ण", "త":"त", "థ":"थ", "ద":"द", "ధ":"ध", "న":"न",
    "ప":"प", "ఫ":"फ", "బ":"ब", "భ":"भ", "మ":"म", "య":"य", "ర":"र", "ల":"ल", "వ":"व", "ళ":"ळ",
    "శ":"श", "ష":"ष", "స":"स", "హ":"ह",
    "ా":"ा", "ి":"ि", "ీ":"ी", "ు":"ु", "ూ":"ू", "ృ":"ृ", "ౄ":"ॄ",
    "ె":"े", "ే":"े", "ై":"ै", "ొ":"ो", "ో":"ो", "ౌ":"ौ",
    "ం":"ं", "ః":"ः", "్":"्",
    "౦":"०", "౧":"१", "౨":"२", "౩":"३", "౪":"४", "౫":"५", "౬":"६", "౭":"७", "౮":"८", "౯":"९"
}

MAPPINGS_TO_DEVANAGARI["Tamil"] = {
    "அ":"अ", "ஆ":"आ", "இ":"इ", "ஈ":"ई", "உ":"उ", "ஊ":"ऊ",
    "எ":"ए", "ஏ":"ए", "ஐ":"ऐ", "ஒ":"ओ", "ஓ":"ओ", "ஔ":"औ",
    "க":"क", "ங":"ङ", "ச":"च", "ஜ":"ज", "ஞ":"ञ", "ட":"ट", "ண":"ण", "த":"त",
    "ந":"न", "ன":"न", "ப":"प", "ம":"म", "ய":"य", "ர":"र", "ற":"र", "ல":"ल",
    "ள":"ळ", "வ":"వ", "ழ":"ळ", "ஶ":"श", "ஷ":"ष", "ஸ":"स", "ஹ":"ह",
    "ா":"ा", "ி":"ि", "ீ":"ी", "ு":"ु", "ூ":"ू", "ெ":"े", "ே":"े", "ை":"ै",
    "ொ":"ो", "ோ":"ो", "ௌ":"ौ", "்":"्", "ஂ":"ं",
    "௦":"०", "௧":"१", "௨":"२", "௩":"३", "௪":"४", "௫":"५", "௬":"६", "௭":"७", "௮":"८", "௯":"९"
}

MAPPINGS_TO_DEVANAGARI["Kannada"] = {
    "ಅ":"अ", "ಆ":"आ", "ಇ":"इ", "ಈ":"ई", "ಉ":"उ", "ಊ":"ऊ", "ಋ":"ऋ", "ೠ":"ॠ",
    "ಎ":"ए", "ಏ":"ए", "ಐ":"ऐ", "ಒ":"ओ", "ಓ":"ओ", "ಔ":"औ",
    "ಕ":"क", "ఖ":"ख", "ಗ":"ग", "ಘ":"घ", "ಙ":"ङ", "ಚ":"च", "ಛ":"छ", "ಜ":"ज", "ಝ":"झ", "ಞ":"ञ",
    "ಟ":"ट", "ಠ":"ठ", "ಡ":"ड", "ಢ":"ढ", "ಣ":"ण", "ತ":"त", "ಥ":"थ", "ದ":"द", "ಧ":"ध", "ನ":"न",
    "ಪ":"प", "ಫ":"फ", "ಬ":"ब", "ಭ":"भ", "ಮ":"म", "ಯ":"य", "ರ":"र", "ಲ":"ल", "ವ":"వ",
    "ಶ":"श", "ಷ":"ष", "ಸ":"स", "ಹ":"ह", "ಳ":"ळ",
    "ಾ":"ा", "ಿ":"ि", "ೀ":"ी", "ು":"ु", "ೂ":"ू", "ೃ":"ृ", "ೆ":"े", "ೇ":"े", "ೈ":"ै",
    "ೊ":"ो", "ೋ":"ो", "ೌ":"ौ", "ಂ":"ं", "ಃ":"ः", "್":"्",
    "೦":"०", "೧":"१", "೨":"२", "೩":"३", "೪":"४", "೫":"५", "೬":"६", "೭":"৭", "೮":"८", "೯":"९"
}

MAPPINGS_TO_DEVANAGARI["Malayalam"] = {
    "അ":"अ", "ആ":"आ", "ഇ":"इ", "ഈ":"ई", "ഉ":"उ", "ഊ":"ऊ", "ഋ":"ऋ", "ൠ":"ॠ", "ഌ":"ऌ", "ൡ":"ॡ",
    "എ":"ए", "ഏ":"ए", "ഐ":"ऐ", "ഒ":"ओ", "ഓ":"ओ", "ഔ":"औ",
    "ക":"क", "ഖ":"ख", "ഗ":"ग", "ഘ":"घ", "ങ":"ङ", "ച":"च", "ഛ":"छ", "ജ":"ज", "ഝ":"झ", "ഞ":"ञ",
    "ട":"ट", "ഠ":"ठ", "ഡ":"ड", "ഢ":"ढ", "ണ":"ण", "ത":"त", "ഥ":"थ", "ദ":"द", "ധ":"ध", "ന":"न",
    "പ":"प", "ഫ":"फ", "ബ":"ब", "ഭ":"भ", "മ":"म", "യ":"य", "ര":"र", "ല":"ल", "വ":"వ",
    "ശ":"श", "ഷ":"ष", "സ":"स", "ഹ":"ह", "ള":"ळ", "ഴ":"ळ", "റ":"र",
    "ാ":"ा", "ി":"ि", "ീ":"ी", "ു":"ु", "ൂ":"ू", "ൃ":"ृ", "െ":"े", "േ":"े", "ൈ":"ै",
    "ൊ":"ो", "ോ":"ो", "ൗ":"ौ", "ം":"ं", "ഃ":"ः", "്":"्",
    "൦":"०", "൧":"१", "൨":"२", "൩":"३", "൪":"४", "൫":"५", "൬":"६", "൭":"७", "൮":"८", "൯":"९"
}

MAPPINGS_TO_DEVANAGARI["Gurmukhi"] = {
    "ਅ":"अ", "ਆ":"आ", "ਇ":"इ", "ਈ":"ई", "ਉ":"उ", "ਊ":"ऊ", "ਏ":"ए", "ਐ":"ऐ", "ਓ":"ओ", "ਔ":"औ",
    "ਕ":"क", "ਖ":"ख", "ਗ":"ग", "ਘ":"घ", "ਙ":"ङ", "ਚ":"च", "ਛ":"छ", "ਜ":"ज", "ਝ":"झ", "ਞ":"ञ",
    "ਟ":"ट", "ਠ":"ठ", "ਡ":"ड", "ਢ":"ढ", "ਣ":"ण", "ਤ":"त", "ਥ":"थ", "ਦ":"द", "ਧ":"ध", "ਨ":"न",
    "ਪ":"प", "ਫ":"फ", "ਬ":"ब", "ਭ":"भ", "ਮ":"म", "ਯ":"य", "ਰ":"र", "ਲ":"ल", "ਵ":"వ", "ੜ":"ड़",
    "ਸ਼":"श", "ਸ":"स", "ਹ":"ह",
    "ਾ":"ा", "ਿ":"ि", "ੀ":"ी", "ੁ":"ु", "ੂ":"ू", "ੇ":"े", "ੈ":"ै", "ੋ":"ो", "ੌ":"ौ", "ਂ":"ं", "਼":"़", "੍":"्",
    "੦":"०", "੧":"१", "੨":"२", "੩":"३", "੪":"४", "੫":"५", "੬":"६", "੭":"७", "੮":"८", "੯":"९"
}

MAPPINGS_TO_DEVANAGARI["Bengali"] = {
    "অ":"अ", "আ":"आ", "ই":"इ", "ঈ":"ई", "উ":"उ", "ঊ":"ऊ", "ঋ":"ऋ", "ৠ":"ॠ",
    "এ":"ए", "ঐ":"ऐ", "ও":"ओ", "ঔ":"औ",
    "ক":"क", "খ":"ख", "গ":"ग", "ঘ":"घ", "ঙ":"ङ", "চ":"च", "ছ":"छ", "জ":"ज", "ঝ":"झ", "ঞ":"ञ",
    "ট":"ट", "ঠ":"ठ", "ড":"ड", "ঢ":"ढ", "ণ":"ण", "ত":"त", "থ":"थ", "দ":"द", "ধ":"ध", "ন":"न",
    "প":"प", "ফ":"फ", "ব":"ब", "ভ":"भ", "ম":"म", "য":"य", "র":"र", "ল":"ल",
    "শ":"श", "ষ":"ष", "স":"स", "হ":"ह", "ড়":"ड़", "ঢ়":"ढ़", "য়":"य",
    "া":"ा", "ি":"ि", "ী":"ी", "ু":"ु", "ূ":"ू", "ৃ":"ृ", "ে":"े", "ৈ":"ै", "ো":"ो", "ৌ":"ौ",
    "ং":"ं", "ঃ":"ः", "্":"्", "ৎ":"त्",
    "০":"०", "১":"१", "২":"२", "৩":"३", "৪":"৪", "৫":"५", "৬":"६", "৭":"৭", "৮":"৮", "৯":"९"
}

MAPPINGS_TO_DEVANAGARI["Odia"] = {
    "ଅ":"अ", "ଆ":"आ", "ଇ":"इ", "ଈ":"ई", "ଉ":"उ", "ଊ":"ऊ", "ଋ":"ऋ", "ୠ":"ॠ",
    "ଏ":"ए", "ଐ":"ऐ", "ଓ":"ओ", "ଔ":"औ",
    "କ":"क", "ଖ":"ख", "ଗ":"ग", "ଘ":"घ", "ଙ":"ङ", "ଚ":"च", "ଛ":"छ", "ଜ":"ज", "ଝ":"झ", "ଞ":"ञ",
    "ଟ":"ट", "ଠ":"ठ", "ଡ":"ड", "ଢ":"ढ", "ଣ":"ण", "ତ":"त", "ଥ":"थ", "ଦ":"द", "ଧ":"ध", "ନ":"न",
    "ପ":"प", "ଫ":"फ", "ବ":"ब", "ଭ":"भ", "ମ":"म", "ଯ":"य", "ର":"र", "ଲ":"ल", "ଳ":"ळ", "ଵ":"व",
    "ଶ":"श", "ଷ":"ष", "ସ":"स", "ହ":"ह", "ଡ଼":"ड़", "ଢ଼":"ढ़", "ୟ":"य",
    "ା":"ा", "ି":"ि", "ୀ":"ी", "ୁ":"ु", "ୂ":"ू", "ୃ":"ृ", "େ":"े", "ୈ":"ै", "ୋ":"ो", "ୌ":"ौ",
    "ଂ":"ं", "ଃ":"ः", "୍":"्",
    "୦":"०", "୧":"१", "୨":"२", "୩":"३", "୪":"४", "୫":"५", "୬":"६", "୭":"७", "୮":"୮", "୯":"९"
}

MAPPINGS_TO_DEVANAGARI["Gujarati"] = {
    "અ":"अ", "આ":"आ", "ઇ":"इ", "ઈ":"ई", "ઉ":"उ", "ઊ":"ऊ", "ઋ":"ऋ",
    "એ":"ए", "ઐ":"ऐ", "ઓ":"ओ", "ઔ":"औ",
    "ક":"क", "ખ":"ख", "ગ":"ग", "ઘ":"घ", "ઙ":"ङ", "ચ":"च", "છ":"छ", "જ":"ज", "ઝ":"झ", "ઞ":"ञ",
    "ટ":"ट", "ઠ":"ठ", "ડ":"ड", "ઢ":"ढ", "ણ":"ण", "ત":"त", "થ":"थ", "દ":"द", "ધ":"ध", "ન":"न",
    "પ":"प", "ફ":"फ", "બ":"ब", "ભ":"भ", "મ":"म", "ય":"य", "ર":"र", "લ":"ल", "વ":"వ",
    "શ":"श", "ષ":"ष", "સ":"स", "હ":"ह", "ળ":"ळ",
    "ા":"ा", "િ":"ि", "ી":"ी", "ુ":"ु", "ૂ":"ू", "ૃ":"ृ", "ે":"े", "ૈ":"ै", "ો":"ो", "ૌ":"ौ",
    "ં":"ं", "ઃ":"ः", "્":"्",
    "૦":"०", "૧":"१", "૨":"૨", "૩":"३", "૪":"૪", "૫":"૫", "૬":"૬", "૭":"૭", "૮":"૮", "૯":"९"
}

# NOTE: English to Devanagari is highly complex and rule-based is not ideal.
# This is a very basic, flawed mapping for demonstration only.
MAPPINGS_TO_DEVANAGARI["English"] = {}


# -----------------------------------------------------
# MAPPINGS: FROM DEVANAGARI TO OTHER SCRIPTS (AUTO-GENERATED)
# -----------------------------------------------------
MAPPINGS_FROM_DEVANAGARI = {}

def generate_reverse_mappings():
    """Generates the reverse mappings from Devanagari to other scripts."""
    approximations = {
        'Tamil': {
            'ख':'க', 'ग':'க', 'घ':'க', 'छ':'ச', 'झ':'ஜ', 'ठ':'ட', 'ड':'ட', 'ढ':'ட',
            'थ':'த', 'द':'த', 'ध':'த', 'फ':'ப', 'ब':'ப', 'भ':'ப', 'श':'ஸ', 'ळ': 'ள'
        }
    }
    for script, mapping in MAPPINGS_TO_DEVANAGARI.items():
        if script == "English": continue # Skip reverse mapping for English
        reverse_mapping = {v: k for k, v in mapping.items()}
        if script in approximations:
            for dev_char, script_char in approximations[script].items():
                if dev_char not in reverse_mapping:
                    reverse_mapping[dev_char] = script_char
        MAPPINGS_FROM_DEVANAGARI[script] = reverse_mapping

generate_reverse_mappings()


# -----------------------------------------------------------------
# NEW: LANGUAGE-SPECIFIC POST-PROCESSING RULES
# -----------------------------------------------------------------
def apply_language_rules(text, language):
    """Applies a set of language-specific orthographic rules."""
    if language == "Hindi":
        # Rule 1: Schwa Deletion at the end of words
        # e.g., "राम" not "रामा"
        words = text.split(' ')
        processed_words = []
        for word in words:
            # Check if the word ends with a Devanagari consonant.
            if word and 0x0915 <= ord(word[-1]) <= 0x0939:
                processed_words.append(word + '्')
            else:
                processed_words.append(word)
        text = ' '.join(processed_words)

        # Rule 2: Heuristic for common character replacements
        # e.g., Prefer 'ल' over 'ळ' in standard Hindi
        text = text.replace('ळ', 'ल')

    # Add rules for other languages here, e.g., if language == "Marathi": ...
    return text

# ---------------------------
# UPGRADED TRANSLITERATION FUNCTION
# ---------------------------
def transliterate(text, target_script, source_script=None, target_language=None):
    """
    Transliterates text, applying exception checks and language-specific rules.
    """
    # --- UPGRADE 1: Check the exception dictionary first ---
    if text.lower() in EXCEPTION_DICTIONARY:
        if target_script in EXCEPTION_DICTIONARY[text.lower()]:
            return EXCEPTION_DICTIONARY[text.lower()][target_script]

    if not source_script:
        source_script = detect_script(text)
        if source_script == "Unknown":
            return f"❌ Could not detect script."

    if source_script == target_script:
        return text

    if source_script == "English":
        return "[English conversion not supported by rules]"

    # Step A: Convert source text to Devanagari hub
    devanagari_text = text
    if source_script != "Devanagari":
        if source_script in MAPPINGS_TO_DEVANAGARI:
            mapping = MAPPINGS_TO_DEVANAGARI[source_script]
            devanagari_text = "".join([mapping.get(char, char) for char in text])
        else:
            return f"❌ No mapping for {source_script} -> Devanagari"

    # Step B: Convert Devanagari hub to target script
    result_text = devanagari_text
    if target_script != "Devanagari":
        if target_script in MAPPINGS_FROM_DEVANAGARI:
            mapping = MAPPINGS_FROM_DEVANAGARI[target_script]
            result_text = "".join([mapping.get(char, char) for char in devanagari_text])
        else:
            return f"❌ No mapping for Devanagari -> {target_script}"

    # --- UPGRADE 2: Apply language-specific post-processing rules ---
    if target_language:
        result_text = apply_language_rules(result_text, target_language)

    return result_text

# ---------------------------
# DEMO / MAIN EXECUTION BLOCK
# ---------------------------
if __name__ == "__main__":
    print("=" * 60)
    print("      Comprehensive Indic Script Transliteration Demo (v2.0)")
    print("=" * 60)

    # --- Combined samples to demonstrate all features ---
    samples = [
        ("నమస్కారం, ప్రపంచం!", "Devanagari", "Telugu", "Hindi"),
        ("मेरा नाम मोहन है", "Telugu", "Devanagari", None),
        ("ಬೆಂಗಳೂರು ಕರ್ನಾಟಕದ ರಾಜಧಾನಿ", "Telugu", "Kannada", None),
        ("কলকাতা ভারতের সাংস্কৃতিক রাজধানী", "Devanagari", "Bengali", "Hindi"),
        ("LipiSathi", "Devanagari", "English", None), # Will be handled by exception dict
        ("రామ", "Devanagari", "Telugu", "Hindi"), # Demonstrates Schwa Deletion
    ]

    for text, target_script, source_script, target_lang in samples:
        transliterated_text = transliterate(text, target_script, source_script, target_lang)
        source = source_script or detect_script(text)
        print(f"({source} -> {target_script} with '{target_lang}' rules)")
        print(f"  Source:       {text}")
        print(f"  Result:       {transliterated_text}\n")

    # Interactive Demo
    while True:
        try:
            user_input = input("\nEnter a word to transliterate (or Ctrl+C to exit): ")
            if not user_input.strip(): continue

            detected_script = detect_script(user_input)
            if detected_script == "Unknown":
                print("❌ Script not detected.")
                continue

            print(f"\nDetected Script: {detected_script}")
            print("-" * 25)
            print(f"Transliterations of '{user_input}':")

            all_scripts = sorted(list(SCRIPT_RANGES.keys()))
            if "Devanagari" not in all_scripts:
                all_scripts.insert(0, "Devanagari")

            for script in all_scripts:
                if script != detected_script:
                    # For demo, apply 'Hindi' rules when converting to Devanagari
                    target_lang = "Hindi" if script == "Devanagari" else None
                    result = transliterate(user_input, script, detected_script, target_lang)
                    print(f"{script.ljust(12)}: {result}")
        except KeyboardInterrupt:
            print("\nExiting. Goodbye!")
            break
        except Exception as e:
            print(f"An error occurred: {e}")

      Comprehensive Indic Script Transliteration Demo (v2.0)
(Telugu -> Devanagari with 'Hindi' rules)
  Source:       నమస్కారం, ప్రపంచం!
  Result:       नमस्कारं, प्रपंचं!

(Devanagari -> Telugu with 'None' rules)
  Source:       मेरा नाम मोहन है
  Result:       మేరా నామ మోహన హై

(Kannada -> Telugu with 'None' rules)
  Source:       ಬೆಂಗಳೂರು ಕರ್ನಾಟಕದ ರಾಜಧಾನಿ
  Result:       బేంగళూరు కర్నాటకద రాజధాని

(Bengali -> Devanagari with 'Hindi' rules)
  Source:       কলকাতা ভারতের সাংস্কৃতিক রাজধানী
  Result:       कलकाता भारतेर् सांस्कृतिक् राजधानी

(English -> Devanagari with 'None' rules)
  Source:       LipiSathi
  Result:       लिपिसाथी

(Telugu -> Devanagari with 'Hindi' rules)
  Source:       రామ
  Result:       राम्


Enter a word to transliterate (or Ctrl+C to exit): నమస్కారం

Detected Script: Telugu
-------------------------
Transliterations of 'నమస్కారం':
Bengali     : নমস্কারং
Devanagari  : नमस्कारं
Gujarati    : નમસ્કારં
Gurmukhi    : ਨਮਸ੍ਕਾਰਂ
Kannada     : ನಮಸ್ಕಾರಂ
Malayalam   : 